# GAN-CLS + DistilBERT — Kaggle GPU training

**Setup (do once):**
1. Zip your local `flower_project` folder EXCEPT the `flower/` venv (and `models/`, `outputs/`).
2. Kaggle → Datasets → New Dataset → upload the zip. Name it e.g. `flower-project`.
3. Kaggle → New Notebook → Settings → Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
4. Add your dataset to the notebook (right panel → Add Input → your dataset).
5. Run the cells below top to bottom.

Training ~200 epochs takes roughly 1-3 hours on a T4. Adjust `epochs` in the config or the override cell.

In [ ]:
# 1) Copy project files from the attached dataset into the writable working dir.
import shutil, os, glob
src = glob.glob('/kaggle/input/*/**/src', recursive=True)
root = os.path.dirname(src[0]) if src else '/kaggle/input/flower-project'
print('project root in input:', root)
for item in ['src', 'configs']:
    s = os.path.join(root, item)
    if os.path.exists(s):
        shutil.copytree(s, f'/kaggle/working/{item}', dirs_exist_ok=True)
print(os.listdir('/kaggle/working'))

In [ ]:
# 2) Kaggle already has torch+CUDA. Only add the two libs it lacks.
!pip install -q transformers datasets pyyaml
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) (Optional) bump batch size / epochs for the GPU before training.
import yaml
with open('configs/train_config.yaml') as f: cfg = yaml.safe_load(f)
cfg['batch_size'] = 64
cfg['epochs'] = 200
cfg['num_workers'] = 2
with open('configs/train_config.yaml', 'w') as f: yaml.safe_dump(cfg, f)
print(cfg)

In [ ]:
# 4) Train. Checkpoints -> /kaggle/working/models, samples -> /kaggle/working/outputs/samples
!python -m src.train --config configs/train_config.yaml

In [ ]:
# 5) Preview a sample grid, then download models/generator_latest.pth from the
#    notebook Output panel and drop it into your local models/ folder.
from IPython.display import Image as IPImage
import glob
pngs = sorted(glob.glob('/kaggle/working/outputs/samples/*.png'))
IPImage(filename=pngs[-1]) if pngs else print('no samples yet')